In [3]:
import polars as pl

# 1. Peek at the behavior labels
# behavior_labels/individual => name of csv file
label_file = '../data/sensor_data/behavior_labels/individual/C01_0725.csv'
try:
    print("--- BEHAVIOR LABELS ---")
    df_labels = pl.read_csv(label_file, n_rows=5)
    print(df_labels.head())
    print("Columns:", df_labels.columns, "\n")
except Exception as e:
    print(f"Skipped Labels: {e}")
        
# 2. Peek at the IMU (collar) Data
# main_data/immu => name of one csv file
immu_file = '../data/sensor_data/main_data/immu/T01/T01_0721.csv'
try:
    print("--- COLLAR SENSOR (IMMU) ---")
    df_immu = pl.read_csv(immu_file, n_rows=5)
    print(df_immu.head())
    print("Columns:", df_immu.columns, "\n")
except Exception as e:
    print(f"Skipped IMMU: {e}")

# 3. Peek at the Weather / THI
thi_file = '../data/sensor_data/main_data/thi/S01.csv'
#thi_file = '../data/sensor_data/main_data/thi/average.csv' # second category of file
try:
    print("--- WEATHER / THI ---")
    df_thi = pl.read_csv(thi_file, n_rows=5)
    print(df_thi.head())
    print("Columns:", df_thi.columns, "\n")
except Exception as e:
    print(f"Skipped THI: {e}")


--- BEHAVIOR LABELS ---
shape: (5, 3)
┌────────────┬──────────┬──────────┐
│ timestamp  ┆ datetime ┆ behavior │
│ ---        ┆ ---      ┆ ---      │
│ i64        ┆ str      ┆ i64      │
╞════════════╪══════════╪══════════╡
│ 1690261200 ┆ 0:00:00  ┆ 0        │
│ 1690261201 ┆ 0:00:01  ┆ 0        │
│ 1690261202 ┆ 0:00:02  ┆ 0        │
│ 1690261203 ┆ 0:00:03  ┆ 0        │
│ 1690261204 ┆ 0:00:04  ┆ 0        │
└────────────┴──────────┴──────────┘
Columns: ['timestamp', 'datetime', 'behavior'] 

--- COLLAR SENSOR (IMMU) ---
shape: (5, 7)
┌───────────┬──────────────┬──────────────┬──────────────┬──────────┬──────────┬──────────┐
│ timestamp ┆ accel_x_mps2 ┆ accel_y_mps2 ┆ accel_z_mps2 ┆ mag_x_uT ┆ mag_y_uT ┆ mag_z_uT │
│ ---       ┆ ---          ┆ ---          ┆ ---          ┆ ---      ┆ ---      ┆ ---      │
│ f64       ┆ f64          ┆ f64          ┆ f64          ┆ f64      ┆ f64      ┆ f64      │
╞═══════════╪══════════════╪══════════════╪══════════════╪══════════╪══════════╪══════════╡
│ 1

### some extracted insights
1. Data type mismatch
labels use i64 (integer) for the timestamp, while the collar and weather sensors use f64 (float)

2. Frequency mismatch
- collar (IMMU)
Multiple rows per second (HIGH freq.)
- Weather (THI)
1 row every few minutes (LOW freq.)

### conclusions
1. We cannot use the regular SQL-style to join into silver layer.
   We need to use a time-series DE tool: join_asof
   It means for every split-second movement the collar recors, look back in time and grab the ost recent behavior label and most recent weather reading.
    


In [6]:
# same using pandas
import pandas as pd

# step-1 Peek the behavior labels
label_file = '../data/sensor_data/behavior_labels/individual/C01_0725.csv'
try:
    print("--- BEHAVIOR LABELS ---")
    df_labels = pd.read_csv(label_file, nrows=5)
    print(df_labels.head())
    print("Columns:", df_labels.columns.tolist(), "\n")
except Exception as e:
    print(f"Skipped Labels: {e}")

# step-2 Peek at IMMU (collar) Data
immu_file = '../data/sensor_data/main_data/immu/T01/T01_0721.csv'
try:
    print("--- COLLAR SENSOR (IMMU) ---")
    df_immu = pd.read_csv(immu_file, nrows=5)
    print(df_immu.head())
    print("Columns:", df_immu.columns.tolist(), "\n")
except Exception as e:
    print(f"Skipped IMMU: {e}")

# step-3 Peek at the Weather / THI
thi_file = '../data/sensor_data/main_data/thi/S01.csv'
#thi_file = '../data/sensor_data/main_data/thi/average.csv' # second category of file
try:
    print("--- WEATHER / THI ---")
    df_thi = pd.read_csv(thi_file, nrows=5)
    print(df_thi.head())
    print("Columns:", df_thi.columns.tolist(), "\n")
except Exception as e:
    print(f"Skipped THI: {e}")

--- BEHAVIOR LABELS ---
    timestamp datetime  behavior
0  1690261200  0:00:00         0
1  1690261201  0:00:01         0
2  1690261202  0:00:02         0
3  1690261203  0:00:03         0
4  1690261204  0:00:04         0
Columns: ['timestamp', 'datetime', 'behavior'] 

--- COLLAR SENSOR (IMMU) ---
      timestamp  accel_x_mps2  accel_y_mps2  accel_z_mps2  mag_x_uT  mag_y_uT  \
0  1.689961e+09        -2.070        -4.299        -7.678      -0.1     -25.1   
1  1.689961e+09        -4.172        -5.128        -8.306       0.4     -22.2   
2  1.689961e+09        -4.201        -4.716        -7.999       0.4     -22.2   
3  1.689961e+09        -3.995        -4.410        -7.855      -2.8     -21.0   
4  1.689961e+09        -3.344        -5.142        -7.664      -2.8     -21.0   

   mag_z_uT  
0     -38.5  
1     -39.4  
2     -39.4  
3     -41.1  
4     -41.1  
Columns: ['timestamp', 'accel_x_mps2', 'accel_y_mps2', 'accel_z_mps2', 'mag_x_uT', 'mag_y_uT', 'mag_z_uT'] 

--- WEATHER / THI --

### NOTE: Pandas vs. Polars
- Pandas eats the RAM!
we ca nuse pd.mergeasof() but it forceces all the data into RAM first.
Managing millions of IMMU rows with the weather in Pandas, It'll freez the computer surely! :(

- Polars can stream ir seamlessly.
It uses join_asof() and doesn't put all the data into RAM at once.

### consclusion: We use Polars!